In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mahdimasoumian/xauusd-2020-2026")

print("Path to dataset files:", path)

100%|██████████| 6.37M/6.37M [00:01<00:00, 3.73MB/s]

Extracting files...


Path to dataset files: C:\Users\User\.cache\kagglehub\datasets\mahdimasoumian\xauusd-2020-2026\versions\1


In [2]:
import os

for p in os.listdir(path):
    print(p)
    path += "/" + p

XAUUSD2020.csv


In [54]:
import pandas as pd
df = pd.read_csv(path)

In [28]:
print (f"row: {len(df)}")
print (f"column: {len(df.columns)}")

row: 465353
column: 7


In [29]:
df.isnull().sum()

date           0
open           0
high           0
low            0
close          0
tick_volume    0
spread         0
dtype: int64

In [30]:
df.head(5)

,date,open,high,low,close,tick_volume,spread
0,1/2/2020 6:00,1520.26,1520.36,1520.16,1520.26,60,14
1,1/2/2020 6:05,1520.27,1520.27,1520.01,1520.11,37,14
2,1/2/2020 6:10,1520.10,1520.10,1519.90,1519.90,52,14
3,1/2/2020 6:15,1519.90,1520.07,1519.87,1519.90,52,14
4,1/2/2020 6:20,1519.90,1520.08,1519.90,1520.00,61,14


In [31]:
df.describe()

,open,high,low,close,tick_volume,spread
count,465353.000000,465353.000000,465353.000000,465353.000000,465353.000000,465353.000000
mean,2411.840693,2412.908855,2410.740855,2411.844872,579.515652,5.969352
std,928.597220,929.360146,927.784361,928.597935,498.109662,7.502973
min,1453.790000,1456.620000,1451.130000,1453.820000,1.000000,0.000000
25%,1805.940000,1806.500000,1805.390000,1805.940000,250.000000,1.000000
50%,1943.700000,1944.440000,1942.940000,1943.710000,442.000000,5.000000
75%,2681.590000,2682.400000,2680.620000,2681.580000,760.000000,6.000000
max,5588.010000,5595.520000,5581.700000,5587.660000,12951.000000,301.000000


In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 465353 entries, 0 to 465352
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   date         465353 non-null  object 
 1   open         465353 non-null  float64
 2   high         465353 non-null  float64
 3   low          465353 non-null  float64
 4   close        465353 non-null  float64
 5   tick_volume  465353 non-null  int64  
 6   spread       465353 non-null  int64  
dtypes: float64(4), int64(2), object(1)
memory usage: 24.9+ MB


### change feature time to datetime

In [53]:
df["date"] = pd.to_datetime(df["date"], utc=True, errors="coerce")
df = df.dropna(subset=["date"]).reset_index(drop=True)

In [55]:
df.head()

,date,open,high,low,close,tick_volume,spread
0,1/2/2020 6:00,1520.26,1520.36,1520.16,1520.26,60,14
1,1/2/2020 6:05,1520.27,1520.27,1520.01,1520.11,37,14
2,1/2/2020 6:10,1520.10,1520.10,1519.90,1519.90,52,14
3,1/2/2020 6:15,1519.90,1520.07,1519.87,1519.90,52,14
4,1/2/2020 6:20,1519.90,1520.08,1519.90,1520.00,61,14


### create a feature chart history

In [56]:
for num in range(1, 25):
    df[f"t_{num}"] = df["close"].shift(num) - df["open"].shift(num)

In [ ]:
df.isna().sum()

In [59]:
df.dropna(inplace=True)

In [60]:
len(df)

465329

In [61]:
df.head()

,date,open,high,low,close,tick_volume,spread,t_1,t_2,t_3,...,t_15,t_16,t_17,t_18,t_19,t_20,t_21,t_22,t_23,t_24
24,1/2/2020 8:00,1520.20,1520.58,1520.18,1520.58,170,14,0.31,0.34,-0.10,...,-0.10,0.00,0.08,-0.06,-0.18,0.10,0.00,-0.20,-0.16,0.00
25,1/2/2020 8:05,1520.58,1520.81,1520.34,1520.42,137,14,0.38,0.31,0.34,...,0.20,-0.10,0.00,0.08,-0.06,-0.18,0.10,0.00,-0.20,-0.16
26,1/2/2020 8:10,1520.40,1520.70,1520.28,1520.28,143,14,-0.16,0.38,0.31,...,-0.56,0.20,-0.10,0.00,0.08,-0.06,-0.18,0.10,0.00,-0.20
27,1/2/2020 8:15,1520.28,1520.28,1519.88,1520.11,78,14,-0.12,-0.16,0.38,...,-0.32,-0.56,0.20,-0.10,0.00,0.08,-0.06,-0.18,0.10,0.00
28,1/2/2020 8:20,1520.11,1520.26,1519.81,1520.16,118,14,-0.17,-0.12,-0.16,...,0.31,-0.32,-0.56,0.20,-0.10,0.00,0.08,-0.06,-0.18,0.10


### create rsi feature

In [62]:
delta = df["close"].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

In [63]:
avg_gain = gain.rolling(window=14, min_periods=14).mean()
avg_loss = loss.rolling(window=14, min_periods=14).mean()

In [64]:
rs = avg_gain / avg_loss
df["rsi"] = 100 - (100 / (1 + rs))
df.loc[avg_loss == 0, "rsi"] = 100

In [65]:
df.dropna(subset=["rsi"], inplace=True)
df.reset_index(drop=True, inplace=True)

In [66]:
df["rsi"] = df["rsi"].round(1)

In [67]:
df[["date", "close", "rsi"]].head(20)

,date,close,rsi
0,1/2/2020 9:10,1520.39,46.6
1,1/2/2020 9:15,1519.85,41.1
2,1/2/2020 9:20,1519.77,41.9
3,1/2/2020 9:25,1519.57,41.5
4,1/2/2020 9:30,1519.68,42.5
5,1/2/2020 9:35,1520.11,50.3
6,1/2/2020 9:40,1519.79,47.4
7,1/2/2020 9:45,1519.37,43.5
8,1/2/2020 9:50,1519.29,36.3
9,1/2/2020 9:55,1519.11,31.0


### create target / Y feature

In [68]:
df["target"] = df["close"].shift(-1) - df["open"].shift(-1)

In [69]:
df.dropna(inplace=True)

In [70]:
df.head(3)

,date,open,high,low,close,tick_volume,spread,t_1,t_2,t_3,...,t_17,t_18,t_19,t_20,t_21,t_22,t_23,t_24,rsi,target
0,1/2/2020 9:10,1520.09,1520.53,1519.95,1520.39,422,14,-0.48,-0.38,0.32,...,-0.10,-0.19,-0.16,0.42,-0.03,0.02,0.21,-0.11,46.6,-0.54
1,1/2/2020 9:15,1520.39,1520.52,1519.79,1519.85,273,14,0.30,-0.48,-0.38,...,0.34,-0.10,-0.19,-0.16,0.42,-0.03,0.02,0.21,41.1,-0.08
2,1/2/2020 9:20,1519.85,1520.06,1519.57,1519.77,250,14,-0.54,0.30,-0.48,...,0.31,0.34,-0.10,-0.19,-0.16,0.42,-0.03,0.02,41.9,-0.20


## Save Dataset

In [71]:
df.to_csv("gold_5m.csv", index=False)

In [72]:
df.head()

,date,open,high,low,close,tick_volume,spread,t_1,t_2,t_3,...,t_17,t_18,t_19,t_20,t_21,t_22,t_23,t_24,rsi,target
0,1/2/2020 9:10,1520.09,1520.53,1519.95,1520.39,422,14,-0.48,-0.38,0.32,...,-0.10,-0.19,-0.16,0.42,-0.03,0.02,0.21,-0.11,46.6,-0.54
1,1/2/2020 9:15,1520.39,1520.52,1519.79,1519.85,273,14,0.30,-0.48,-0.38,...,0.34,-0.10,-0.19,-0.16,0.42,-0.03,0.02,0.21,41.1,-0.08
2,1/2/2020 9:20,1519.85,1520.06,1519.57,1519.77,250,14,-0.54,0.30,-0.48,...,0.31,0.34,-0.10,-0.19,-0.16,0.42,-0.03,0.02,41.9,-0.20
3,1/2/2020 9:25,1519.77,1520.02,1519.18,1519.57,221,14,-0.08,-0.54,0.30,...,0.38,0.31,0.34,-0.10,-0.19,-0.16,0.42,-0.03,41.5,0.19
4,1/2/2020 9:30,1519.49,1520.31,1519.06,1519.68,432,14,-0.20,-0.08,-0.54,...,-0.16,0.38,0.31,0.34,-0.10,-0.19,-0.16,0.42,42.5,0.43
